# 🎬 Malaak Video Maker

Make short vertical videos from your phone. Free.

**How to use:**
1. Sign in with your Google account.
2. Tap the ▶️ play button on the big cell below.
3. If it says *"This notebook was not authored by Google"*, tap **Run anyway**.
4. Wait 3-5 minutes. Then tap the big blue link.
5. Type your topic or script and tap make.

**Keys (optional):** A free Pexels key gives moving stock videos. A free Gemini key writes better scripts. You can leave them empty.

⚠️ **Download your videos before you close this page.** When the session ends, all videos are deleted.

Each free session lasts a few hours. Next time, just tap play again.

For true stories, check the facts and name your source.

In [ ]:
#@title ▶️ Tap play to start Malaak Video Maker
#@markdown Optional keys. Leave empty if you don't have them.
PEXELS_API_KEY = "" #@param {type:"string"}
GEMINI_API_KEY = "" #@param {type:"string"}
import os, subprocess
print("Setting up... this takes about 3-5 minutes. Please wait.")
subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg imagemagick > /dev/null", shell=True)
if not os.path.exists("/content/malaak-video-maker"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/reseachrandomv/malaak-video-maker", "/content/malaak-video-maker"], check=True)
os.chdir("/content/malaak-video-maker")
r = subprocess.run("pip install -q -r requirements.txt 2>&1 | tail -5", shell=True)
os.environ["PEXELS_API_KEY"] = PEXELS_API_KEY.strip()
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY.strip()
os.environ["IMAGEMAGICK_BINARY"] = "/usr/bin/convert"
%run scripts/colab_start.py


## 📺 Auto-post to YouTube (one-time setup, ~10 min)
1. Open **console.cloud.google.com** → tap **Select a project → New project** → name it `malaak-video` → Create.
2. Search **YouTube Data API v3** → tap **Enable**.
3. Go to **APIs & Services → OAuth consent screen** → External → app name `Malaak Video Maker`, your email → Save. Under **Audience / Test users** add your own Gmail.
4. Go to **Credentials → Create credentials → OAuth client ID** → type **TVs and Limited Input devices** → Create.
5. Copy the **Client ID** and **Client secret** into the boxes below and tap ▶️.
6. A code appears. Open **google.com/device**, type the code, pick your YouTube account, tap Allow.

After that, every finished video posts by itself as a public Short. Don't share these values with anyone.


In [ ]:
#@title 🔑 Sign in to YouTube (run after the first cell)
YT_CLIENT_ID = "" #@param {type:"string"}
YT_CLIENT_SECRET = "" #@param {type:"string"}
SAVE_LOGIN_TO_MY_DRIVE = False #@param {type:"boolean"}
PRIVACY = "public" #@param ["public", "unlisted", "private"]
import os, shutil, sys
os.environ["YT_PRIVACY"] = PRIVACY
drive_tok = "/content/drive/MyDrive/MalaakVideoMaker/yt_token.json"
if SAVE_LOGIN_TO_MY_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
sys.path.insert(0, "/content/malaak-video-maker")
from app.services import youtube_upload as yt
if SAVE_LOGIN_TO_MY_DRIVE and os.path.exists(drive_tok) and not YT_CLIENT_ID.strip():
    shutil.copy(drive_tok, yt.TOKEN_FILE); print("✅ Signed in from your Drive.")
else:
    from IPython.display import HTML, display
    def show(url, code):
        display(HTML(f'<h2>1. Open <a href="{url}" target="_blank">{url}</a></h2><h1>2. Type: {code}</h1><p>Waiting for you to tap Allow...</p>'))
    yt.device_login(YT_CLIENT_ID.strip(), YT_CLIENT_SECRET.strip(), show)
    print("✅ YouTube connected! New videos will post automatically.")
    if SAVE_LOGIN_TO_MY_DRIVE:
        os.makedirs(os.path.dirname(drive_tok), exist_ok=True); shutil.copy(yt.TOKEN_FILE, drive_tok)
        print("Saved to your Drive. Next time tick the box, leave ID/secret empty, and tap play.")


In [ ]:
#@title 🤖 Autopilot: make and post videos in a row (keep this tab open)
HOW_MANY_VIDEOS = 3 #@param {type:"integer"}
#@markdown Your own topics, separated by `;` (e.g. `Dyatlov Pass; Mary Celeste`). Leave empty and Gemini picks true crime (world + Maldives), real stories and mysteries.
TOPICS = "" #@param {type:"string"}
VOICE = "en-US-GuyNeural-Male" #@param ["en-US-GuyNeural-Male", "en-US-JennyNeural-Female", "en-US-AriaNeural-Female", "en-GB-RyanNeural-Male"]
import os, sys
os.chdir("/content/malaak-video-maker"); sys.path.insert(0, os.getcwd())
from scripts.autopilot import run
run(int(HOW_MANY_VIDEOS), [t for t in TOPICS.split(";")], VOICE, os.environ.get("YT_PRIVACY", "public"))
print("\n🎉 Autopilot finished.")


Based on [MoneyPrinterTurbo](https://github.com/harry0703/MoneyPrinterTurbo) by harry0703 (MIT License).